In [ ]:
"""
SINGLE-STAGE CONV-TASNET PIPELINE (Variable Speaker Count: 2 to 5)
Optimized for Kaggle / Single GPU.

This architecture uses Temporal Convolutional Networks (TCN) and a Dynamic
Permutation Invariant Training (PIT) loss. It forces the network to output 5 
streams; active speakers are scored via SI-SDR, and inactive streams are 
forced to silence via an L2 energy penalty.
"""
import os
import re
import glob
import json
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader, Subset
from scipy.optimize import linear_sum_assignment
import numpy as np

# config

CONFIG = {
    # data
    "data_root": "/kaggle/input/datasets/viditarora18/speech-sep-mixtures",
    "train_split_name": "train",
    "val_split_name": "dev",
    "sample_rate": 8000,
    "segment_seconds": 2.0,
    "batch_size": 8,  # Conv-TasNet is lighter, can safely use 8 for training
    "num_workers": 2,
    "max_train_samples_per_bucket": 500,
    "max_val_samples_per_bucket": 40,
    "max_speakers": 5,

    # Conv-TasNet Hyperparameters 
    "N": 256,  # Number of filters in autoencoder
    "L": 16,   # Length of filters in autoencoder (2ms at 8kHz)
    "B": 128,  # Number of channels in bottleneck and residual paths
    "H": 256,  # Number of channels in convolutional blocks
    "P": 3,    # Number of repeats of TCN blocks
    "X": 8,    # Number of convolutional blocks in each repeat
    
    # training
    "epochs": 100,      # Single curriculum, train across everything
    "lr": 1e-3,         # Higher LR for faster convergence
    "grad_clip": 5.0,
    "silence_weight": 100.0, # Multiplier to aggressively zero-out inactive channels
    "log_every": 50,
    "checkpoint_dir": "/kaggle/working/checkpoints",
    "session_time_budget_hours": 8.0, 

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
}

# data

def find_split_dir(data_root, split, n_spk):
    pattern = os.path.join(data_root, "**", f"{split}_mix{n_spk}")
    candidates = [p for p in glob.glob(pattern, recursive=True) if os.path.isdir(p)]
    if not candidates: return None
    return max(candidates, key=lambda p: p.count(os.sep))

def index_samples(split_dir, n_spk):
    items = []
    subfolders = [d for d in glob.glob(os.path.join(split_dir, "*")) if os.path.isdir(d)]
    for d in sorted(subfolders):
        mix_candidates = glob.glob(os.path.join(d, "mix*.wav"))
        if not mix_candidates: continue
        mix_path = mix_candidates[0]
        src_paths = [os.path.join(d, f"s{i + 1}.wav") for i in range(n_spk)]
        if all(os.path.exists(p) for p in src_paths):
            items.append({"mix": mix_path, "sources": src_paths, "n_spk": n_spk})
    if items: return items
    mix_files = sorted(glob.glob(os.path.join(split_dir, "mix*.wav")))
    for mix_path in mix_files:
        m = re.search(r"(\d+)", os.path.basename(mix_path))
        if not m: continue
        sample_id = m.group(1)
        src_paths = [glob.glob(os.path.join(split_dir, f"s{i + 1}*{sample_id}*.wav")) for i in range(n_spk)]
        if all(src_paths):
            items.append({"mix": mix_path, "sources": [p[0] for p in src_paths], "n_spk": n_spk})
    return items

class SepDataset(Dataset):
    def __init__(self, data_root, split, min_spk=2, max_spk=5, sample_rate=8000, 
                 segment_seconds=2.0, train=True, max_samples=None):
        self.sr = sample_rate
        self.seg_len = int(segment_seconds * sample_rate)
        self.train = train
        self.items = []
        rng = random.Random(0)
        for n in range(min_spk, max_spk + 1):
            d = find_split_dir(data_root, split, n)
            if d is None: continue
            found = index_samples(d, n)
            if max_samples and len(found) > max_samples:
                found = rng.sample(found, max_samples)
            self.items.extend(found)
        print(f"[data] split='{split}' TOTAL: {len(self.items)} mixtures")

    def __len__(self):
        return len(self.items)

    def _load(self, path):
        wav, sr = torchaudio.load(path)
        if sr != self.sr:
            wav = torchaudio.functional.resample(wav, sr, self.sr)
        return wav.mean(dim=0)

    def __getitem__(self, idx):
        item = self.items[idx]
        mix = self._load(item["mix"])
        sources = [self._load(p) for p in item["sources"]]
        common_len = min([mix.shape[0]] + [s.shape[0] for s in sources])
        mix = mix[:common_len]
        sources = [s[:common_len] for s in sources]
        
        if self.train:
            if common_len > self.seg_len:
                start = random.randint(0, common_len - self.seg_len)
                mix = mix[start:start + self.seg_len]
                sources = [s[start:start + self.seg_len] for s in sources]
            else:
                pad = self.seg_len - common_len
                mix = F.pad(mix, (0, pad))
                sources = [F.pad(s, (0, pad)) for s in sources]
        return {"mix": mix, "sources": torch.stack(sources, dim=0), "n_spk": item["n_spk"]}

def collate_fn(batch):
    T = batch[0]["mix"].shape[0]
    mixes = torch.stack([b["mix"] for b in batch], dim=0)
    n_spks = torch.tensor([b["n_spk"] for b in batch])
    max_spk = max(n_spks)
    sources = torch.zeros(len(batch), max_spk, T)
    for i, b in enumerate(batch):
        sources[i, :b["n_spk"]] = b["sources"]
    return {"mix": mixes, "sources": sources, "n_spk": n_spks}

# model

class GlobalLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(1, dim, 1))
        self.bias = nn.Parameter(torch.zeros(1, dim, 1))

    def forward(self, x):
        mean = torch.mean(x, (1, 2), keepdim=True)
        var = torch.mean((x - mean)**2, (1, 2), keepdim=True)
        x = (x - mean) / torch.sqrt(var + self.eps)
        return x * self.weight + self.bias

class Conv1DBlock(nn.Module):
    def __init__(self, in_channels, conv_channels, kernel_size, dilation):
        super().__init__()
        self.conv1x1 = nn.Conv1d(in_channels, conv_channels, 1)
        self.prelu1 = nn.PReLU()
        self.norm1 = GlobalLayerNorm(conv_channels)
        
        padding = (kernel_size - 1) * dilation // 2
        self.depthwise = nn.Conv1d(conv_channels, conv_channels, kernel_size, 
                                   dilation=dilation, padding=padding, groups=conv_channels)
        self.prelu2 = nn.PReLU()
        self.norm2 = GlobalLayerNorm(conv_channels)
        
        self.res_conv = nn.Conv1d(conv_channels, in_channels, 1)
        self.skip_conv = nn.Conv1d(conv_channels, in_channels, 1)

    def forward(self, x):
        y = self.norm1(self.prelu1(self.conv1x1(x)))
        y = self.norm2(self.prelu2(self.depthwise(y)))
        res = self.res_conv(y)
        skip = self.skip_conv(y)
        return x + res, skip

class ConvTasNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.max_spk = cfg["max_speakers"]
        self.N, self.L = cfg["N"], cfg["L"]
        
        # Encoder
        self.encoder = nn.Conv1d(1, self.N, self.L, stride=self.L // 2, bias=False)
        self.enc_norm = GlobalLayerNorm(self.N)
        self.bottleneck = nn.Conv1d(self.N, cfg["B"], 1)

        # Separation TCN
        self.tcn = nn.ModuleList()
        for p in range(cfg["P"]):
            for x in range(cfg["X"]):
                dilation = 2**x
                self.tcn.append(Conv1DBlock(cfg["B"], cfg["H"], 3, dilation))
        
        # Mask Generation
        self.mask_conv = nn.Conv1d(cfg["B"], self.max_spk * self.N, 1)
        
        # Decoder
        self.decoder = nn.ConvTranspose1d(self.N, 1, self.L, stride=self.L // 2, bias=False)

    def forward(self, mix):
        B, T = mix.shape
        x = mix.unsqueeze(1)
        
        enc = F.relu(self.encoder(x))
        y = self.enc_norm(enc)
        y = self.bottleneck(y)
        
        skip_connection = 0
        for i, layer in enumerate(self.tcn):
            y, skip = layer(y)
            skip_connection = skip_connection + skip
            
        masks = self.mask_conv(skip_connection)
        masks = F.relu(masks).view(B, self.max_spk, self.N, -1)
        
        masked_enc = enc.unsqueeze(1) * masks
        
        # Decode all max_spk streams
        masked_enc = masked_enc.view(B * self.max_spk, self.N, -1)
        decoded = self.decoder(masked_enc).squeeze(1)
        
        # Ensure output length matches input exactly
        if decoded.shape[-1] > T: decoded = decoded[..., :T]
        elif decoded.shape[-1] < T: decoded = F.pad(decoded, (0, T - decoded.shape[-1]))
            
        return decoded.view(B, self.max_spk, T)

# losses

def si_sdr(estimate, target, eps=1e-8):
    target = target - target.mean(dim=-1, keepdim=True)
    estimate = estimate - estimate.mean(dim=-1, keepdim=True)
    dot = torch.sum(estimate * target, dim=-1, keepdim=True)
    energy = torch.sum(target ** 2, dim=-1, keepdim=True) + eps
    s_target = dot / energy * target
    e_noise = estimate - s_target
    ratio = torch.sum(s_target ** 2, dim=-1) / (torch.sum(e_noise ** 2, dim=-1) + eps)
    return 10 * torch.log10(ratio + eps)

def dynamic_pit_loss(estimates, targets, n_spk, cfg):
    """
    Computes standard PIT for the active speakers, and applies an L2 energy
    penalty to the unused output channels to force them to be completely silent.
    """
    B, max_out, T = estimates.shape
    total_sdr_loss = 0.0
    total_silence_loss = 0.0
    
    estimates_np = estimates.detach().cpu().numpy()
    
    for b in range(B):
        m = n_spk[b].item()
        est = estimates[b]  # [5, T]
        tgt = targets[b, :m] # [m, T]
        
        # Calculate pairwise SI-SDR for all 5 outputs against the 'm' real targets
        sdr_matrix = torch.zeros(max_out, m, device=est.device)
        for i in range(max_out):
            for j in range(m):
                sdr_matrix[i, j] = si_sdr(est[i].unsqueeze(0), tgt[j].unsqueeze(0)).squeeze()
        
        # Hungarian Algorithm to find best assignment
        cost_matrix = -sdr_matrix.detach().cpu().numpy()
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        
        # Accumulate SI-SDR loss for matched active speakers
        matched_sdr = sdr_matrix[row_ind, col_ind].mean()
        total_sdr_loss += -matched_sdr
        
        # Accumulate Silence Loss for unused outputs
        unassigned = [idx for idx in range(max_out) if idx not in row_ind]
        if unassigned:
            silence_penalty = (est[unassigned] ** 2).mean()
            total_silence_loss += silence_penalty
            
    avg_sdr_loss = total_sdr_loss / B
    avg_silence_loss = total_silence_loss / B
    
    total_loss = avg_sdr_loss + (cfg["silence_weight"] * avg_silence_loss)
    
    # Convert to float to safely handle cases where total_silence_loss remains a python 0.0
    return total_loss, float(-avg_sdr_loss), float(avg_silence_loss)

# resume/utils

def load_state(checkpoint_dir):
    path = os.path.join(checkpoint_dir, "run_state.json")
    if not os.path.exists(path): return None
    with open(path) as f: return json.load(f)

def save_state(checkpoint_dir, state):
    os.makedirs(checkpoint_dir, exist_ok=True)
    path = os.path.join(checkpoint_dir, "run_state.json")
    with open(path + ".tmp", "w") as f: json.dump(state, f)
    os.replace(path + ".tmp", path) 

def save_checkpoint(checkpoint_dir, name, model, optimizer):
    os.makedirs(checkpoint_dir, exist_ok=True)
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict()}, 
               os.path.join(checkpoint_dir, name))

def load_checkpoint(checkpoint_dir, name, model, optimizer):
    path = os.path.join(checkpoint_dir, name)
    if not os.path.exists(path): return False
    payload = torch.load(path, map_location="cpu")
    model.load_state_dict(payload["model"])
    optimizer.load_state_dict(payload["optimizer"])
    return True

# main loop

def main():
    cfg = CONFIG
    torch.manual_seed(cfg["seed"])
    ckpt_dir = cfg["checkpoint_dir"]
    use_amp = (cfg["device"] == "cuda")
    run_start = time.time()

    print("[data] Loading datasets...")
    train_ds = SepDataset(cfg["data_root"], cfg["train_split_name"], 2, 5, 
                          cfg["sample_rate"], cfg["segment_seconds"], train=True, 
                          max_samples=cfg["max_train_samples_per_bucket"])
    val_ds = SepDataset(cfg["data_root"], cfg["val_split_name"], 2, 5, 
                        cfg["sample_rate"], cfg["segment_seconds"], train=False, 
                        max_samples=cfg["max_val_samples_per_bucket"])
    
    # Shuffle False for val to keep comparisons identical across epochs
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, 
                              collate_fn=collate_fn, num_workers=cfg["num_workers"], drop_last=True)
    
    # batch_size=1 for validation to handle variable length natural audio
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, 
                            collate_fn=collate_fn, num_workers=cfg["num_workers"])

    model = ConvTasNet(cfg).to(cfg["device"])
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    state = load_state(ckpt_dir)
    if state is None:
        state = {"epoch": 0, "best_val_sdr": float("-inf")}
        print("\n[run] Fresh Start")
    else:
        print(f"\n[run] RESUMING from Epoch {state['epoch']}")
        load_checkpoint(ckpt_dir, "latest.pt", model, optimizer)

    global_step = 0

    while state["epoch"] < cfg["epochs"]:
        if (time.time() - run_start) > (cfg["session_time_budget_hours"] * 3600 - 300):
            print(f"\n[budget] Reached time limit cleanly at epoch {state['epoch']}. Exiting.")
            return

        epoch_t0 = time.time()
        print(f"\n--- Epoch {state['epoch']} ---")
        
        # train
        model.train()
        epoch_sdr = 0
        for batch in train_loader:
            optimizer.zero_grad()
            mix = batch["mix"].to(cfg["device"])
            sources = batch["sources"].to(cfg["device"])
            n_spk = batch["n_spk"]
            
            with torch.autocast(device_type="cuda" if use_amp else "cpu", enabled=use_amp):
                estimates = model(mix)
                loss, sdr, sil = dynamic_pit_loss(estimates, sources, n_spk, cfg)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            
            epoch_sdr += sdr
            global_step += 1
            if global_step % cfg["log_every"] == 0:
                print(f"  step {global_step} | Loss: {loss.item():.2f} | SI-SDR: {sdr:.2f}dB | Silence Penalty: {sil:.4f}")

        # validate
        model.eval()
        val_sdr = 0
        val_batches = 0
        with torch.no_grad():
            for batch in val_loader:
                mix = batch["mix"].to(cfg["device"])
                sources = batch["sources"].to(cfg["device"])
                n_spk = batch["n_spk"]
                estimates = model(mix)
                _, sdr, _ = dynamic_pit_loss(estimates, sources, n_spk, cfg)
                val_sdr += sdr
                val_batches += 1
        
        avg_val_sdr = val_sdr / val_batches
        print(f"  [val] SI-SDR: {avg_val_sdr:.2f}dB (Time: {time.time() - epoch_t0:.0f}s)")
        
        # save
        save_checkpoint(ckpt_dir, "latest.pt", model, optimizer)
        if avg_val_sdr > state["best_val_sdr"]:
            state["best_val_sdr"] = avg_val_sdr
            save_checkpoint(ckpt_dir, "best.pt", model, optimizer)
            print(f"  -> NEW BEST MODEL! ({avg_val_sdr:.2f}dB)")
            
        state["epoch"] += 1
        save_state(ckpt_dir, state)

    print("\n[run] TRAINING COMPLETE!")

main()